In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
test_labels = pd.read_csv('../../../results/PlantCAD2_tasks/exp-leaf-abs/test.tsv', sep='\t')

In [3]:
model_dict = [
    ('pcv2_1', 'pcv2-l24-d0768-checkpoints-lr-1e-4'),
    ('pcv2_2', 'pcv2-l48-d1024-checkpoints-lr-1e-4'),
    ('pcv2_3', 'pcv2-l48-d1536-checkpoints-lr-1e-4'),
    ('agront', 'agront-checkpoints-lr-1e-4'),
    ('sup_pcv2_1', 'sup_pcv2-l24-d0768-checkpoints-lr-1e-4'),
    ('cnn_lstm', 'cnn_lstm'),
]

In [4]:
for model in model_dict:
    curPATH = f'../../../results/PlantCAD2_tasks/exp-leaf-abs/{model[1]}/predictions.csv'
    curPreds = pd.read_csv(curPATH, sep='\t')
    test_labels[f'{model[0]}'] = curPreds['predicted_value']

In [5]:
model_names = [
    'pcv2_1', 'pcv2_2', 'pcv2_3', 'agront', 'sup_pcv2_1', 'cnn_lstm'
]
long_df = test_labels.melt(
    id_vars=['orthogroup', 'Label'],
    value_vars=model_names,
    var_name='model',
    value_name='scores'
)

In [6]:
def safe_spearman(x, y):
    if pd.Series(x).nunique() < 2 or pd.Series(y).nunique() < 2:
        return np.nan
    return spearmanr(x, y).correlation

scc_og = (
    long_df
      .groupby(['model', 'orthogroup'])
      .apply(lambda g: pd.Series({
          'scc': safe_spearman(g['scores'], g['Label']),
          'count': len(g)
      }))
      .reset_index()
)

# Filter small orthogroups
scc_og = scc_og[scc_og['count'] > 20]

In [7]:
summary = (
    scc_og
      .groupby('model')['scc']
      .agg(mean='mean', median='median', n='count')
      .reset_index()
      .sort_values('mean', ascending=False)
)
print(summary)

        model      mean    median      n
4      pcv2_3  0.156687  0.138558  19556
2      pcv2_1  0.156453  0.140255  19556
3      pcv2_2  0.156084  0.138466  19556
0      agront  0.147443  0.129592  19556
5  sup_pcv2_1  0.127504  0.112891  19556
1    cnn_lstm  0.126199  0.112475  19556


In [8]:
scc_og[scc_og['orthogroup'] == '3I2A2']

,model,orthogroup,scc,count
0,agront,3I2A2,0.264369,154.0
21932,cnn_lstm,3I2A2,0.652248,154.0
43864,pcv2_1,3I2A2,0.168318,154.0
65796,pcv2_2,3I2A2,0.408072,154.0
87728,pcv2_3,3I2A2,0.284690,154.0
109660,sup_pcv2_1,3I2A2,0.488570,154.0


In [9]:
order = model_names
order

['pcv2_1', 'pcv2_2', 'pcv2_3', 'agront', 'sup_pcv2_1', 'cnn_lstm']

In [10]:
scc_og.head()

,model,orthogroup,scc,count
0,agront,3I2A2,0.264369,154.0
1,agront,3I2A4,-0.189458,59.0
2,agront,3I2A5,0.621367,25.0
3,agront,3I2A6,0.015562,52.0
4,agront,3I2A7,0.011978,26.0


In [11]:
scc_og.to_csv('og_level_scc.tsv', sep='\t', index=False)

In [ ]:
plt.figure(figsize=(6, 5))
sns.violinplot(
    data=scc_og, x='model', y='scc',
    order=order, inner='box', cut=0, linewidth=1,
    palette='tab10'
)

# overlay mean (orange diamond)
means = summary.set_index('model')['mean']
xpos = np.arange(len(order))
plt.scatter(xpos, means.loc[order].values, marker='D', s=50, zorder=3, c="orange", label="Mean")

# overlay median (black horizontal line)
medians = summary.set_index('model')['median']
for i, model in enumerate(order):
    plt.hlines(
        y=medians[model],
        xmin=i - 0.3, xmax=i + 0.3, 
        colors='red', linestyles='-', lw=2, label="Median" if i == 0 else ""
    )

plt.xlabel('Model')
plt.ylabel('Per-orthogroup Spearman (scc)')
plt.title('Gene expression: per-orthogroup scc by model')
plt.xticks(rotation=30, ha='right')
plt.legend()
plt.tight_layout()
plt.show()